In [ ]:
from etils import ecolab
import jax

with ecolab.adhoc(
    source="fig-codelab",
    reload='tunix',
    behavior='preferred',
    cell_autoreload=True,
):
  from tunix.models.gemma4 import model as tunix_model
  from tunix.models.gemma4.params import map_from_upstream_checkpoint
  from flax import nnx
  from gemma.gm.nn.gemma4 import _config as up_config
  from gemma.gm.nn.gemma4 import _layers
  from gemma.gm.nn.gemma4 import _modules
  from gemma.gm.nn.gemma4 import _transformer as up_transformer
  import jax
  import jax.numpy as jnp
  import numpy as np



In [ ]:
config = tunix_model.ModelConfig(
    num_layers=4,
    num_embed=1000,
    embed_dim=128,
    hidden_dim=256,
    num_heads=4,
    head_dim=32,
    num_kv_heads=1,
    per_layer_input_dim=32,
    sliding_window_size=32,
    query_pre_attn_norm=tunix_model.QueryPreAttentionNormalisation.NONE,
)

pattern = (
    # tunix_model.AttentionType.LOCAL_SLIDING,
    tunix_model.AttentionType.GLOBAL,
)
up_pattern = []
for i in range(config.num_layers):
  p = pattern[i % len(pattern)]
  if p == tunix_model.AttentionType.LOCAL_SLIDING:
    up_pattern.append(_modules.AttentionType.LOCAL_SLIDING)
  elif p == tunix_model.AttentionType.GLOBAL:
    up_pattern.append(_modules.AttentionType.GLOBAL)

up_cfg = up_config.TransformerConfig(
    num_embed=config.num_embed,
    embed_dim=config.embed_dim,
    hidden_dim=config.hidden_dim,
    num_heads=config.num_heads,
    head_dim=config.head_dim,
    num_kv_heads=config.num_kv_heads,
    use_post_attn_norm=True,
    use_post_ffw_norm=True,
    attention_types=tuple(up_pattern),
    sliding_window_size=config.sliding_window_size,
    global_rope_proportion=1.0,
    local_rope_proportion=1.0,
    final_logit_softcap=None,
    per_layer_input_dim=config.per_layer_input_dim,
)

upstream = up_transformer.Transformer(config=up_cfg)

rng = jax.random.PRNGKey(0)
T = 64  # pylint: disable=invalid-name
tokens = jax.random.randint(rng, (2, T), 0, config.num_embed)
init_vars = upstream.init(rng, tokens)

mapped_params = map_from_upstream_checkpoint(init_vars['params'])

rngs = nnx.Rngs(42)
model = tunix_model.Gemma4(config, rngs=rngs)
nnx.update(model, mapped_params)

positions = jnp.tile(
    jnp.arange(tokens.shape[1])[None, :], (tokens.shape[0], 1)
)
causal_mask = jnp.tile(
    jnp.tril(jnp.ones((T, T), dtype=jnp.bool_))[None, ...], (2, 1, 1)
)

x_tunix = model.embedder.encode(tokens)
per_layer_input = model.embedder.encode_per_layer_input(x_tunix, tokens)

def trace_upstream(self, t, mask, per_layer_in):
  x = self.embedder.encode(t)
  res = []
  for i, b in enumerate(self.blocks):
    _, x = b(
        x, positions, None, mask, per_layer_input=per_layer_in[..., i, :]
    )
    res.append(x)
  return res

up_trace = upstream.apply(
    init_vars, tokens, causal_mask, per_layer_input, method=trace_upstream
)

for i, layer in enumerate(model.layers):
  _, x_tunix = layer(
      x_tunix,
      positions,
      None,
      causal_mask,
      per_layer_input=per_layer_input[..., i, :],
  )
  print(
      f'\n[LAYER {i}] Tunix mean: {jnp.mean(x_tunix)}, Upstream mean:'
      f' {jnp.mean(up_trace[i])}'
  )
  np.testing.assert_allclose(x_tunix, up_trace[i], rtol=1e-3, atol=1e-3)

print("\nAll Matched!!!")

In [ ]:
# # RMSNorm ALL MATCHED!!!

# rng = nnx.Rngs(42)
# tunix_norm = tunix_model.RMSNorm(64, rngs=rngs)
# up_norm = _layers.RMSNorm()

# x = jax.random.normal(rngs.params(), (4, 128, 64))
# variables = up_norm.init(rngs.params(), x)

# tunix_norm.scale.value = variables['params']['scale']

# tunix_out = tunix_norm(x)
# up_out = up_norm.apply(variables, x)

# np.testing.assert_allclose(tunix_out, up_out, rtol=1e-5, atol=1e-5)

In [ ]:
# # Attention Module. ALL MATCHED!!!

# class DummyConfig:
#   num_q_heads = 4
#   num_kv_heads = 1
#   embed_dim = 64
#   head_dim = 16
#   param_dtype = jnp.float32
#   sliding_window_size = 32
#   rope_base_frequency = 10000
#   rope_scale_factor = 1.0
#   query_pre_attn_norm = tunix_model.QueryPreAttentionNormalisation.NONE
#   remat_config = tunix_model.RematConfig.NONE

#   shd_config = tunix_model.ShardingConfig.get_default_sharding()

# config = DummyConfig()
# rngs = nnx.Rngs(42)

# tunix_attn = tunix_model.Attention(
#     num_heads=config.num_q_heads,
#     num_kv_heads=config.num_kv_heads,
#     features=config.embed_dim,
#     head_dim=config.head_dim,
#     attn_type=tunix_model.AttentionType.LOCAL_SLIDING,
#     rngs=rngs,
#     sliding_window_size=config.sliding_window_size,
#     rope_base_frequency=config.rope_base_frequency,
#     rope_scale_factor=config.rope_scale_factor,
#     query_pre_attn_norm=config.query_pre_attn_norm,
#     shd_config=config.shd_config,
#     remat_config=config.remat_config,
#     param_dtype=config.param_dtype,
# )

# up_attn = _modules.Attention(
#     num_heads=config.num_q_heads,
#     num_kv_heads=config.num_kv_heads,
#     features=config.embed_dim,
#     key_size=config.head_dim,
#     attn_type=_modules.AttentionType.LOCAL_SLIDING,
#     rope_proportion=1.0,
#     sliding_window_size=config.sliding_window_size,
# )

# x = jax.random.normal(rngs.params(), (4, 128, config.embed_dim))
# segment_pos = jnp.tile(jnp.arange(128, dtype=jnp.int32), (4, 1))
# attn_mask = jnp.tril(jnp.ones((4, 128, 128)), k=0).astype(jnp.bool_)

# variables = up_attn.init(
#     rngs.params(), x, segment_pos, None, attn_mask
# )

# tunix_attn.q_einsum.w.value = variables['params']['q_einsum']['w']
# tunix_attn.kv_einsum.w.value = variables['params']['kv_einsum']['w']
# tunix_attn._query_norm.scale.value = variables['params']['query_norm'][
#     'scale'
# ]
# tunix_attn._key_norm.scale.value = variables['params']['key_norm']['scale']
# tunix_attn.attn_vec_einsum.w.value = variables['params']['attn_vec_einsum'][
#     'w'
# ]

# _, tunix_out = tunix_attn.block(x, segment_pos, None, attn_mask)
# _, up_out = up_attn.apply(variables, x, segment_pos, None, attn_mask)

# np.testing.assert_allclose(tunix_out, up_out, rtol=1e-5, atol=1e-5)

In [ ]:
# # MLP ALL MATCHED!!!

# class DummyConfig:
#   embed_dim = 64
#   hidden_dim = 128
#   param_dtype = jnp.float32

# config = DummyConfig()
# rngs = nnx.Rngs(42)

# tunix_ff = tunix_model.FeedForward(config=config, rngs=rngs)

# up_ff = _modules.FeedForward(
#     features=config.embed_dim,
#     hidden_dim=config.hidden_dim,
# )

# x = jax.random.normal(rngs.params(), (4, 128, config.embed_dim))

# variables = up_ff.init(rngs.params(), x)

# up_gate_w = variables['params']['gating_einsum']
# tunix_ff.gate_proj.kernel.value = up_gate_w[0].T
# tunix_ff.up_proj.kernel.value = up_gate_w[1].T
# tunix_ff.down_proj.kernel.value = variables['params']['linear']

# tunix_out = tunix_ff(x)
# up_out = up_ff.apply(variables, x)

# np.testing.assert_allclose(tunix_out, up_out, rtol=1e-5, atol=1e-5)

In [ ]:
# # DecoderLayer MATCHED!!!

# class DummyConfig:
#   num_heads = 4
#   num_kv_heads = 1
#   embed_dim = 64
#   head_dim = 16
#   hidden_dim = 128
#   param_dtype = jnp.float32
#   sliding_window_size = 16
#   c = 10_000
#   local_base_frequency = 10_000
#   global_base_frequency = 1_000_000
#   global_rope_proportion = 0.25
#   local_rope_proportion = 1.0
#   local_scale_factor = 1.0
#   global_scale_factor = 1.0
#   query_pre_attn_norm = tunix_model.QueryPreAttentionNormalisation.NONE
#   remat_config = tunix_model.RematConfig.NONE
#   enable_moe = False
#   per_layer_input_dim = 0
#   num_global_kv_heads = None
#   global_key_size = None
#   k_eq_v_global = False
#   global_rope_proportion = 0.25
#   local_rope_proportion = 1.0

#   shd_config = tunix_model.ShardingConfig.get_default_sharding()

# config = DummyConfig()
# rngs = nnx.Rngs(42)
# attn_type = tunix_model.AttentionType.GLOBAL
# up_attn_type = _modules.AttentionType.GLOBAL if attn_type == tunix_model.AttentionType.GLOBAL else _modules.AttentionType.LOCAL_SLIDING

# tunix_layer = tunix_model.DecoderLayer(
#     config=config,
#     attn_type=attn_type,
#     rngs=rngs,
# )

# up_layer = _modules.Block(
#     num_heads=config.num_heads,
#     num_kv_heads=config.num_kv_heads,
#     embed_dim=config.embed_dim,
#     head_dim=config.head_dim,
#     hidden_dim=config.hidden_dim,
#     use_post_attn_norm=True,
#     use_post_ffw_norm=True,
#     attn_type=up_attn_type,
#     rope_base_frequency=config.global_base_frequency if up_attn_type == _modules.AttentionType.GLOBAL else config.local_base_frequency,
#     sliding_window_size=config.sliding_window_size,
#     global_rope_proportion=config.global_rope_proportion,
#     local_rope_proportion=config.local_rope_proportion,
# )

# x = jax.random.normal(rngs.params(), (4, 128, config.embed_dim))
# segment_pos = jnp.tile(jnp.arange(128, dtype=jnp.int32), (4, 1))
# attn_mask = jnp.tril(jnp.ones((4, 128, 128)), k=0).astype(jnp.bool_)

# variables = up_layer.init(
#     rngs.params(), x, segment_pos, None, attn_mask
# )

# # Map weights
# tunix_layer.pre_attention_norm.scale.value = variables['params'][
#     'pre_attention_norm'
# ]['scale']

# up_attn = variables['params']['attn']
# tunix_layer.attn.q_einsum.w.value = up_attn['q_einsum']['w']
# tunix_layer.attn.kv_einsum.w.value = up_attn['kv_einsum']['w']
# tunix_layer.attn._query_norm.scale.value = up_attn['query_norm']['scale']
# tunix_layer.attn._key_norm.scale.value = up_attn['key_norm']['scale']
# tunix_layer.attn.attn_vec_einsum.w.value = up_attn['attn_vec_einsum']['w']

# tunix_layer.post_attention_norm.scale.value = variables['params'][
#     'post_attention_norm'
# ]['scale']

# tunix_layer.pre_ffw_norm.scale.value = variables['params']['pre_ffw_norm'][
#     'scale'
# ]

# up_gate_w = variables['params']['mlp']['gating_einsum']
# tunix_layer.mlp.gate_proj.kernel.value = up_gate_w[0].T
# tunix_layer.mlp.up_proj.kernel.value = up_gate_w[1].T
# tunix_layer.mlp.down_proj.kernel.value = variables['params']['mlp'][
#     'linear'
# ]

# tunix_layer.post_ffw_norm.scale.value = variables['params'][
#     'post_ffw_norm'
# ]['scale']

# tunix_layer.skip_scale.value = variables['params']['skip_scale']

# _, tunix_out = tunix_layer(x, segment_pos, None, attn_mask)
# _, up_out= up_layer.apply(variables, x, segment_pos, None, attn_mask)

# # np.testing.assert_allclose(tunix_in_norm, up_in_norm, rtol=1e-5, atol=1e-5)
# # np.testing.assert_allclose(tunix_ori_attn, up_ori_attn, rtol=1e-5, atol=1e-5)
# # np.testing.assert_allclose(tunix_attn, up_attn, rtol=1e-5, atol=1e-5)
# np.testing.assert_allclose(tunix_out, up_out, rtol=1e-5, atol=1e-5)

In [ ]:
config = tunix_model.ModelConfig(
    num_layers=4,
    num_embed=1000,
    num_heads = 4,
    num_kv_heads = 1,
    embed_dim = 64,
    head_dim = 16,
    hidden_dim = 128,
    param_dtype = jnp.float32,
    sliding_window_size = 16,
    local_base_frequency = 10_000,
    global_base_frequency = 1_000_000,
    global_rope_proportion = 0.25,
    local_rope_proportion = 1.0,
    local_scale_factor = 1.0,
    global_scale_factor = 1.0,
    enable_moe = False,
    per_layer_input_dim = 0,
    num_global_kv_heads = None,
    global_key_size = None,
    k_eq_v_global = False,
)

pattern = (
    tunix_model.AttentionType.LOCAL_SLIDING,
    tunix_model.AttentionType.GLOBAL,
)
up_pattern = []
for i in range(config.num_layers):
  p = pattern[i % len(pattern)]
  if p == tunix_model.AttentionType.LOCAL_SLIDING:
    up_pattern.append(_modules.AttentionType.LOCAL_SLIDING)
  elif p == tunix_model.AttentionType.GLOBAL:
    up_pattern.append(_modules.AttentionType.GLOBAL)

up_cfg = up_config.TransformerConfig(
    num_embed=config.num_embed,
    embed_dim=config.embed_dim,
    hidden_dim=config.hidden_dim,
    num_heads=config.num_heads,
    head_dim=config.head_dim,
    num_kv_heads=config.num_kv_heads,
    use_post_attn_norm=True,
    use_post_ffw_norm=True,
    attention_types=tuple(up_pattern),
    sliding_window_size=config.sliding_window_size,
    global_rope_proportion=1.0,
    local_rope_proportion=1.0,
    final_logit_softcap=None,
    per_layer_input_dim=config.per_layer_input_dim,
)

upstream = up_transformer.Transformer(config=up_cfg)

rng = jax.random.PRNGKey(0)
T = 64  # pylint: disable=invalid-name
tokens = jax.random.randint(rng, (2, T), 0, config.num_embed)
init_vars = upstream.init(rng, tokens)

mapped_params = map_from_upstream_checkpoint(init_vars['params'])

rngs = nnx.Rngs(42)
model = tunix_model.Gemma4(config, rngs=rngs)
nnx.update(model, mapped_params)

positions = jnp.tile(
    jnp.arange(tokens.shape[1])[None, :], (tokens.shape[0], 1)
)
causal_mask = jnp.tile(
    jnp.tril(jnp.ones((T, T), dtype=jnp.bool_))[None, ...], (2, 1, 1)
)

x_tunix = model.embedder.encode(tokens)
per_layer_input = model.embedder.encode_per_layer_input(x_tunix, tokens)

def trace_upstream(self, t, mask, per_layer_in):
  x = self.embedder.encode(t)
  res = []
  for i, b in enumerate(self.blocks):
    _, x = b(
        x, positions, None, mask, per_layer_input=per_layer_in[..., i, :]
    )
    res.append(x)
  return res

up_trace = upstream.apply(
    init_vars, tokens, causal_mask, per_layer_input, method=trace_upstream
)

for i, layer in enumerate(model.layers):
  _, x_tunix = layer(
      x_tunix,
      positions,
      None,
      causal_mask,
      per_layer_input=per_layer_input[..., i, :],
  )
  print(
      f'\n[LAYER {i}] Tunix mean: {jnp.mean(x_tunix)}, Upstream mean:'
      f' {jnp.mean(up_trace[i])}'
  )
  np.testing.assert_allclose(x_tunix, up_trace[i], rtol=1e-3, atol=1e-3)